# Refine and Customize Model Helpers

In the previous lessons we built baselines: the LLM picked 3 classifiers, we scored each with defaults, and kept the winner. That's a starting point.

In this lesson we refine those baselines with **Optuna** (Bayesian hyperparameter search). We don't need the LLM here — the candidates are already chosen. We define a small search space per model and let Optuna tune all three via cross-validation. The tuned leaderboard picks the final winner, and sometimes it's not the same model that won at defaults.

We'll walk through this with classification, but the exact same pattern applies to regression. Just swap `classification_helper` for `regression_helper`, ROC AUC for RMSE, and you're done.

## 1 - Setup

The completed notebook includes saved outputs so you can review the expected result without my local `.env` file. To rerun the Gemini cells, create your own `.env` file in the project root with `GEMINI_API_KEY=your-gemini-api-key-here`.


In [ ]:
%pip install -q google-genai pandas scikit-learn python-dotenv optuna

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("../02-Preprocessing-Helpers").resolve()))
sys.path.append(str(Path("../03-Modeling-Helpers").resolve()))

In [ ]:
import warnings

import optuna
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score

from preprocessing_pipeline import preprocessing_pipeline
from classification_helper import classification_helper

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
result = preprocessing_pipeline(
    raw_path="../../data/hr_analytics.csv",
    target_col="Attrition",
    task="classification",
)
X_train = result.X_train_enc
X_test = result.X_test_enc
y_train = result.y_train
y_test = result.y_test


## 2 - Run the 3.1 Baseline (Before)

Use the saved helper to get the untuned baseline. This is our "before" number.

> **Note:** We walk through this with classification, but the same pattern applies to regression — swap `classification_helper` for `regression_helper`, `StratifiedKFold` for `KFold`, ROC AUC for RMSE, and adjust the search spaces for regressors. Everything else stays the same.

In [ ]:
baseline = classification_helper(X_train, X_test, y_train, y_test)

baseline_auc = roc_auc_score(y_test, baseline["y_proba"])
print(f"Baseline winner: {baseline['model_name']}")
print(f"Baseline test ROC AUC: {baseline_auc:.3f}")
print(f"Init kwargs: {baseline['init_kwargs']}")

## 3 - Define Search Spaces

We already have our 3 candidates from the baseline — no need to ask the LLM again. We define a small hardcoded search space per model (3–5 tunable params) and hand it to Optuna.

In [ ]:
SEARCH_SPACES = {
    "LogisticRegression": lambda trial: {
        "C": trial.suggest_float("C", 1e-3, 1e2, log=True),
        "penalty": trial.suggest_categorical("penalty", ["l1", "l2"]),
        "solver": "liblinear",
        "class_weight": "balanced",
        "random_state": 42,
    },
    "RandomForestClassifier": lambda trial: {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=50),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "class_weight": "balanced",
        "random_state": 42,
    },
    "XGBClassifier": lambda trial: {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "eval_metric": "logloss",
        "random_state": 42,
    },
}

candidates = baseline["candidates"]
for c in candidates:
    print(
        f"- {c['class']}: {'defined' if c['class'] in SEARCH_SPACES else 'no search space — will skip'}"
    )

## 4 - Tune Each Candidate with Optuna

For each candidate, run 10 trials × 3-fold CV on ROC AUC and rank by the best tuned score.

In [ ]:
def tune(candidate, X_train, y_train, n_trials=10):
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    cls = candidate["cls"]
    space_fn = SEARCH_SPACES[candidate["class"]]

    def objective(trial):
        clf = cls(**space_fn(trial))
        return cross_val_score(
            clf, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1
        ).mean()

    study = optuna.create_study(
        direction="maximize", sampler=optuna.samplers.TPESampler(seed=42)
    )
    study.optimize(objective, n_trials=n_trials)

    # Rebuild best params by running the space_fn on a FrozenTrial-like wrapper
    best_params = space_fn(optuna.trial.FixedTrial(study.best_params))
    return {
        "model": f"{candidate['module']}.{candidate['class']}",
        "cls": cls,
        "roc_auc": study.best_value,
        "best_params": best_params,
    }

In [ ]:
leaderboard = (
    pd.DataFrame([tune(c, X_train, y_train) for c in candidates])
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

leaderboard[["model", "roc_auc", "best_params"]]

## 5 - Refit the Winner and Compare (After)

In [ ]:
winner = leaderboard.iloc[0]
tuned_model = winner["cls"](**winner["best_params"])
tuned_model.fit(X_train, y_train)
tuned_proba = tuned_model.predict_proba(X_test)[:, 1]
tuned_auc = roc_auc_score(y_test, tuned_proba)

pd.DataFrame(
    [
        {
            "stage": "baseline (3.1)",
            "model": baseline["model_name"],
            "test_roc_auc": baseline_auc,
        },
        {"stage": "tuned (3.3)", "model": winner["model"], "test_roc_auc": tuned_auc},
    ]
)